# M6 Lab — pandas (cleaning, groupby, merge)

**Datasets:** `titanic_clean.csv`, `ecommerce_orders.csv` &nbsp;|&nbsp; **Anchors:** McKinney Ch7–10, Géron Ch2

No `[AI-OFF]` cells in this lab. Oral checkpoint at end of M6.

## L6.1 — Missing data `[McKinney Ch7 p.215–230]`

In [ ]:
import pandas as pd, numpy as np
np.random.seed(42)
df = pd.read_csv('titanic_clean.csv')

missing = df.isna().sum().to_frame('n_missing')
missing['pct'] = (missing['n_missing'] / len(df) * 100).round(1)
missing.sort_values('pct', ascending=False)

## L6.2 — `groupby()` agg / transform / filter `[McKinney Ch10 p.290–325]`

In [ ]:
# Aggregate
df.groupby('class')['fare'].agg(['mean', 'median', 'count'])

# Transform — preserves index, enables row-level computation
df['fare_z'] = df.groupby('class')['fare'].transform(lambda s: (s - s.mean()) / s.std())
df[['class', 'fare', 'fare_z']].head()

## L6.3 — Safe merging `[McKinney Ch8 p.255–275]`

In [ ]:
orders = pd.read_csv('ecommerce_orders.csv')
# Always specify validate= to catch silent row explosion
# merged = orders.merge(customers, on='customer_id', validate='m:1')

## L6.6 — `.pipe()` chains `[McKinney Ch7 p.235–240]`

In [ ]:
def drop_dupes(d): return d.drop_duplicates()
def fill_age(d):
    d = d.copy()
    d['age'] = d['age'].fillna(d['age'].median())
    return d

cleaned = (df.pipe(drop_dupes).pipe(fill_age))
cleaned.shape

## L6.8 — Categorical dtype `[McKinney Ch7 p.245–260]`

In [ ]:
before = df.memory_usage(deep=True).sum() / 1e6
df['class'] = df['class'].astype('category')
after  = df.memory_usage(deep=True).sum() / 1e6
print(f'before={before:.2f} MB  after={after:.2f} MB')

## L6.9 — Imputation strategies `[Géron Ch2 p.72–78]`

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
# Note: imputer must be fit on TRAIN ONLY in modelling workflow (preview of L10.4)

---
## Submission checklist
- [ ] Missing-data table produced
- [ ] At least one `.pipe()` chain
- [ ] Categorical dtype memory comparison printed
- [ ] Oral checkpoint prep notes for `.pipe()` chain explanation

## L6.6 — Synthetic Data Generation [🔮 ai-generated]


### Step 1: Define your schema and generate data

We will ask Gemma 4n to generate realistic e-commerce order data matching an explicit schema.


In [ ]:
import requests, json, pandas as pd

def ask_gemma(prompt: str, model: str = "gemma4n") -> str:
    r = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False
        },
        timeout=120
    )
    r.raise_for_status()
    return r.json()["message"]["content"]

schema_prompt = """
Generate 20 rows of realistic e-commerce order data as a JSON array.
Each row: {"order_id": int, "customer_age": int, "product_category": str,
           "order_value": float, "days_to_delivery": int, "returned": bool}
Constraints: age 18-75, value 5.00-500.00, delivery 1-14 days.
Return ONLY the JSON array, no explanation.
"""

raw = ask_gemma(schema_prompt)
raw[:500]


### Step 2: Parse and inspect [AI-VERIFY]


In [ ]:
df_synth = pd.DataFrame(json.loads(raw))
print(df_synth.dtypes)
df_synth.describe()


### Step 3: Validate with pandera


In [ ]:
import pandera as pa

synth_schema = pa.DataFrameSchema(
    {
        "order_id": pa.Column(int, unique=True, nullable=False, checks=pa.Check.ge(1)),
        "customer_age": pa.Column(int, nullable=False, checks=pa.Check.between(18, 75)),
        "product_category": pa.Column(str, nullable=False),
        "order_value": pa.Column(float, nullable=False, checks=pa.Check.between(5.0, 500.0)),
        "days_to_delivery": pa.Column(int, nullable=False, checks=pa.Check.between(1, 14)),
        "returned": pa.Column(bool, nullable=False),
    },
    strict=True,
)

validated_synth = synth_schema.validate(df_synth, lazy=True)
validated_synth.head()


### Step 4: Compare to real data distributions


In [ ]:
real_orders = pd.read_csv("ecommerce_orders.csv").rename(columns={"category": "product_category"})
shared_numeric = [
    col
    for col in df_synth.columns.intersection(real_orders.columns)
    if pd.api.types.is_numeric_dtype(df_synth[col]) and pd.api.types.is_numeric_dtype(real_orders[col])
]
print("Shared numeric columns:", shared_numeric)

if shared_numeric:
    comparison = pd.concat(
        {
            "synthetic": df_synth[shared_numeric].describe(),
            "real": real_orders[shared_numeric].describe(),
        },
        axis=1,
    )
    comparison
else:
    print("No shared numeric columns found to compare.")


## L6.8 [AI-OFF] — Full wrangling pipeline

**Build this pipeline on your own, without AI assistance.**
Your pipeline must: (1) handle all missing values with justified strategies, (2) compute at least one groupby aggregation, (3) merge two DataFrames with validate=, (4) use .pipe() to chain steps, (5) pass a pandera schema check.
